In [16]:
import pandas as pd
import numpy as np

# STEP 1 — Load CSV
file_path = "loans_full_schema.csv"
data = pd.read_csv(file_path)

# STEP 2 — Risk mapping logic
def map_risk(status):
    if status in ['Current','In Grace Period']:
        return 'Low'
    elif status in ['Late (16-30 days)']:
        return 'Medium'   
    elif status in ['Charged Off', 'Late (31-120 days)']:
        return 'High'
    else:
        return 'Unknown'

data['risk_category'] = data['loan_status'].apply(map_risk)

# STEP 3 — Split Medium and High
medium_data = data[data['risk_category'] == 'Medium']
high_data = data[data['risk_category'] == 'High']

# Number of rows to be added
new_medium_rows = 2000
new_high_rows = 2000

def generate_synthetic_rows(original_df, n_rows):
    """
    Create new synthetic rows based on original data distributions.
    """
    synthetic_rows = []
    
    for _ in range(n_rows):
        row = {}
        for col in original_df.columns:
            if col == 'risk_category':
                continue

            if original_df[col].dtype == 'O':
                # For categoricals, sample from real values
                row[col] = np.random.choice(original_df[col].dropna().values)
            else:
                # For numerics, generate around mean ± std
                series = original_df[col].dropna()
                if len(series) == 0:
                    row[col] = np.nan
                    continue
                
                mean_val = series.mean()
                std_val = series.std()
                min_val = series.min()
                max_val = series.max()

                # Handle case where std = 0
                if std_val == 0 or np.isnan(std_val):
                    new_val = mean_val
                else:
                    new_val = np.random.normal(loc=mean_val, scale=std_val)
                    new_val = np.clip(new_val, min_val, max_val)
                
                row[col] = new_val

        synthetic_rows.append(row)
    
    return pd.DataFrame(synthetic_rows)

# STEP 4 — Generate new Medium rows
new_medium = generate_synthetic_rows(medium_data, new_medium_rows)
new_medium['risk_category'] = 'Medium'

# STEP 5 — Generate new High rows
new_high = generate_synthetic_rows(high_data, new_high_rows)
new_high['risk_category'] = 'High'

# STEP 6 — Combine with original data
augmented_data = pd.concat([data, new_medium, new_high], ignore_index=True)

# STEP 7 — Save to CSV
augmented_data.to_csv("loans_full_schema_augmented.csv", index=False)

print("Successfully created 4000 synthetic rows and saved the augmented CSV.")


Successfully created 4000 synthetic rows and saved the augmented CSV.
